In [ ]:
import os
from pathlib import Path
import pandas as pd
from constants import output_dir, blues, reds, font_prop
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from scipy.stats import gaussian_kde

# NC

## Length

In [ ]:
data_dir = "/Users/zhoukuangqi/Desktop/pepbenchmark/0923_statistic"

color = blues[2]
edge_color = blues[-2]
vline_color = reds[-3]
dataset_names = [each for each in os.listdir(data_dir) if each.startswith("nc-")]

In [ ]:
os.makedirs(save_dir := (output_dir / Path("nc_prop_0923")), exist_ok=True)

In [ ]:
for dataset_name in dataset_names:
    data_info = pd.read_csv(data_dir/Path(dataset_name)/ Path("all.csv"))

    lengths = data_info['length']
    
    len_values, counts = np.unique(lengths, return_counts=True)
    mean = np.mean(lengths)
    med = np.median(len_values)

    if len(counts) > 10:
        plt.figure(figsize=(16, 6))
        plt.bar(
            len_values, 
            counts, 
            color=color, 
            width=1.0, 
            align='center',
            edgecolor=edge_color,
            alpha=0.8
        )
    else:
        plt.figure(figsize=(6, 6))
        plt.bar(
            len_values, 
            counts, 
            color=color, 
            width=1.0, 
            align='center',
            edgecolor=edge_color,
            alpha=0.8
        )
        int_med = int(med)
        
        x_min, x_max = max(0, int_med-10), min(int_med+10, 22)
        plt.xlim(x_min, x_max)
        ticks = np.arange(x_min, x_max, 2)
    
        new_labels = [int(t) for t in ticks]
        plt.xticks(ticks, new_labels)

    plt.axvline(x=mean, color=vline_color, linestyle='--', linewidth=2, label="mean")
    plt.axvline(x=med, color=vline_color, linestyle=':', linewidth=2, label="median")

    plt.xlabel("Number of Amino Acids (Sequence Length)", fontsize=14, fontproperties=font_prop,)
    plt.ylabel("Occurence", fontsize=14, fontproperties=font_prop,)
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)

    legend_font_prop = deepcopy(font_prop)
    legend_font_prop.set_size(14)
    plt.legend(fontsize=20, prop=legend_font_prop)
    plt.grid(True, linestyle='--', color='gray', alpha=0.5)

    plt.savefig(
        save_dir / Path(f"len_{dataset_name}.png"), 
        dpi=200, 
        bbox_inches="tight", 
        transparent=True
    )

## AA

In [ ]:
for dataset_name in dataset_names:
    data_info = pd.read_csv(data_dir/Path(dataset_name)/ Path("monomer_frequency_analysis.csv"))
        
    aa_to_frec = {row['Monomer']: row['Frequency'] for _, row in data_info.iterrows() if row['Type'] == "Standard"}
    
    aa_types = list(sorted(aa_to_frec.keys()))
    counts = [aa_to_frec[aa] for aa in aa_types]

    x = np.arange(len(aa_types))

    plt.figure(figsize=(16, 6))
    
    plt.bar(
        x, 
        counts, 
        color=color, 
        width=1.0, 
        align='center',
        edgecolor=edge_color,
        alpha=0.8
    )

    plt.xticks(x, aa_types)

    plt.xlabel("Amino Acid Type", fontsize=14, fontproperties=font_prop,)
    plt.ylabel("Occurrence", fontsize=14, fontproperties=font_prop,)
    plt.xticks(fontsize=13, fontproperties=font_prop)
    plt.yticks(fontsize=13, fontproperties=font_prop)

    plt.savefig(
        save_dir / Path(f"aa_{dataset_name}.png"), 
        dpi=200, 
        bbox_inches="tight", 
        transparent=True
    )

    plt.title(dataset_name)

## NCAA

In [ ]:
for dataset_name in dataset_names:
    data_info = pd.read_csv(data_dir/Path(dataset_name)/ Path("monomer_frequency_analysis.csv"))
        
    aa_to_frec = {row['Monomer']: row['Frequency'] for _, row in data_info.iterrows() if row['Type'] == "Non-standard"}
    
    aa_types = list(sorted(aa_to_frec.keys(), key=lambda k: aa_to_frec[k], reverse=True))[:20]
    counts = [aa_to_frec[aa] for aa in aa_types]

    origin_aa = data_info[data_info['Type']=='Non-standard'].groupby(
        by="Monomer"
    ).agg(
        {"Natural_Analog": lambda x: ", ".join(x)}
        ).reset_index(drop=False).set_index("Monomer")["Natural_Analog"].to_dict()

    x = np.arange(len(aa_types))

    plt.figure(figsize=(16, 6))
    
    plt.bar(
        x, 
        counts, 
        color=color, 
        width=1.0, 
        align='center',
        edgecolor=edge_color,
        alpha=0.8
    )

    plt.xticks(x, [" ".join([aa_type, f"({origin_aa[aa_type]})"]) for aa_type in aa_types])

    plt.xlabel("Monomer Type", fontsize=14, fontproperties=font_prop,)
    plt.ylabel("Occurrence", fontsize=14, fontproperties=font_prop,)
    plt.xticks(fontsize=13, fontproperties=font_prop, rotation=45)
    plt.yticks(fontsize=13, fontproperties=font_prop)

    plt.savefig(
        save_dir / Path(f"ncaa_{dataset_name}.png"), 
        dpi=200, 
        bbox_inches="tight", 
        transparent=True
    )

    plt.title(dataset_name)

# Prop

In [ ]:
colors = {
    "pos": reds[3],
    "neg": blues[2],
}
edge_colors = {
    "pos": reds[-2],
    "neg": blues[-2],
}
legends = {
    "pos": "positive",
    "neg": "negtive",
}
    
x_axis_names = {
    'seq_len': "Sequence Length",
    'hydro': "Hydrophilicity",
    'charge': "Charge",
    "iso": "isoelectric_point",
}
    
prop_names = {
    "nonfouling": ["seq_len"],
    "ttca": ["seq_len"],
}


def get_prop(prop_name, row):
    if prop_name == "seq_len":
        return row["length"]
    else:
        raise ValueError

In [ ]:
for dataset_name, prop_names_ in prop_names.items():
    props = {prop_name: {
        "pos": [], "neg": []
        } for prop_name in prop_names_
    }
    data_info = pd.read_csv(
        data_dir / Path(f"len_data_{dataset_name}.csv")
    )

    for _, row in data_info.iterrows():
        for prop_name in prop_names[dataset_name]:
            (props[prop_name]["pos"] if row["label"] else props[prop_name]["neg"]).append(
                get_prop(prop_name, row)
            )

    fig, axe = plt.subplots(1, 1, figsize=(16, 6))
    for i, prop_name in enumerate(prop_names[dataset_name]):
        for label in ['pos', 'neg']:
            prop_values = np.asarray(props[prop_name][label])
            p_prop = gaussian_kde(prop_values)

            counts, bins, _ = axe.hist(
                prop_values, 
                bins=50, 
                alpha=0.5, 
                color=colors[label],
                density=False,
                label=legends[label],
            )
            legend_font_prop = deepcopy(font_prop)
            legend_font_prop.set_size(14)
            axe.legend(fontsize=20, prop=legend_font_prop)
            
            bin_width = bins[1] - bins[0]
            y_min, y_max = prop_values.min(), prop_values.max()
            x = np.linspace(y_min, y_max, 500)
            y = p_prop(x) * len(prop_values) * bin_width

            axe.plot(
                x,
                y,
                color=edge_colors[label],
                linewidth=2,
                alpha=0.8,
            )

        axe.set_xlabel(x_axis_names[prop_name], fontsize=14, fontproperties=font_prop)
        if i == 0:
            axe.set_ylabel("Occurrence", fontsize=14, fontproperties=font_prop)
        axe.tick_params(axis='x', labelsize=13)
        axe.tick_params(axis='y', labelsize=13)
        axe.grid(True, linestyle='--', color='gray', alpha=0.5)
    plt.savefig(
        save_dir / Path(f"{dataset_name}.png"),
        dpi=200,
        bbox_inches="tight",
        transparent=True,
    )